# 06 — Destroy: Planned, human-in-the-loop (public-safe)

## Google Drive Setup

In [11]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Check if drive is already mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Drive is already mounted.")

print("Done!")

# Filepath Search
#search_term = "your_filename.ext"  # Change this
search_term = "synthetic_assets.csv"
!find /content/drive/MyDrive -maxdepth 15 -type f -iname "synthetic_assets.csv" -print


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive is already mounted.
Done!
/content/drive/MyDrive/data/synthetic_assets.csv


In [12]:
!pip -q install pandas numpy matplotlib  # safe to re-run

In [13]:
import pandas as pd
assets = pd.read_csv('/content/drive/MyDrive/data/synthetic_assets.csv')
assets['created_date'] = pd.to_datetime(assets['created_date'])
assets['age_days'] = (pd.Timestamp.today() - assets['created_date']).dt.days
assets['eligible_for_destroy'] = assets['age_days'] >= assets['retention_days']
eligible = assets[assets['eligible_for_destroy']].copy()
eligible[['asset_id','sensitivity','age_days','retention_days']].head(10)

,asset_id,sensitivity,age_days,retention_days
0,a000,confidential,667,180
3,a003,internal,479,365
4,a004,confidential,353,180
5,a005,internal,710,365
6,a006,confidential,445,180
8,a008,internal,371,365
9,a009,internal,743,365
10,a010,internal,754,365
11,a011,restricted,633,90
12,a012,restricted,569,90


In [14]:
eligible['status'] = 'pending_approvals'
eligible['required_approvals'] = eligible['sensitivity'].map({
    'restricted': 'data_owner + security + legal',
    'confidential': 'data_owner + security',
    'internal': 'data_owner',
    'public': 'data_owner'
})
eligible['scheduled_for'] = (pd.Timestamp.today() + pd.Timedelta(days=7)).date()
eligible['destroy_action'] = 'crypto_shred_or_logical_delete (pending)'
eligible[['asset_id','sensitivity','status','required_approvals','scheduled_for','destroy_action']].head(12)

,asset_id,sensitivity,status,required_approvals,scheduled_for,destroy_action
0,a000,confidential,pending_approvals,data_owner + security,2026-02-09,crypto_shred_or_logical_delete (pending)
3,a003,internal,pending_approvals,data_owner,2026-02-09,crypto_shred_or_logical_delete (pending)
4,a004,confidential,pending_approvals,data_owner + security,2026-02-09,crypto_shred_or_logical_delete (pending)
5,a005,internal,pending_approvals,data_owner,2026-02-09,crypto_shred_or_logical_delete (pending)
6,a006,confidential,pending_approvals,data_owner + security,2026-02-09,crypto_shred_or_logical_delete (pending)
8,a008,internal,pending_approvals,data_owner,2026-02-09,crypto_shred_or_logical_delete (pending)
9,a009,internal,pending_approvals,data_owner,2026-02-09,crypto_shred_or_logical_delete (pending)
10,a010,internal,pending_approvals,data_owner,2026-02-09,crypto_shred_or_logical_delete (pending)
11,a011,restricted,pending_approvals,data_owner + security + legal,2026-02-09,crypto_shred_or_logical_delete (pending)
12,a012,restricted,pending_approvals,data_owner + security + legal,2026-02-09,crypto_shred_or_logical_delete (pending)


## Audit evidence export

In [15]:
evidence = eligible[['asset_id','sensitivity','required_approvals','scheduled_for','destroy_action']].copy()
evidence.to_csv('/content/drive/MyDrive/data/destroy_intent_register.csv', index=False)
print('Wrote data/destroy_intent_register.csv')

Wrote data/destroy_intent_register.csv
